# Curiosity wheel anomaly detection

Notebook Kaggle autosufficiente. Individua il dataset, configura il preprocessing per modello, prepara i DataLoader e definisce il contratto comune di modelli, metriche e artefatti.

In [ ]:
!pip install -q google-api-python-client google-auth

In [ ]:
from __future__ import annotations

import csv
import hashlib
import json
import os
from collections import Counter
from collections.abc import Iterable, Mapping
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Sequence

import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.nn import functional as F
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision.models import get_model, get_model_weights
from torchvision.models.feature_extraction import create_feature_extractor
from torchvision.transforms import InterpolationMode
from torchvision.transforms import functional as TF
from torchvision.transforms.functional import pil_to_tensor

In [ ]:
# Configuration
KAGGLE_INPUT = Path("/kaggle/input")
BATCH_SIZE = 4
NUM_WORKERS = 2
PIN_MEMORY = torch.cuda.is_available()
SEED = 42

# Model selection and PatchCore parameters.
MODEL_NAME = "patchcore"
PATCHCORE_BACKBONE = "resnet18"
PATCHCORE_PRETRAINED = True
PATCHCORE_LAYERS = ("layer2", "layer3")
PATCHCORE_CORESET_SAMPLING_RATIO = 0.1
PATCHCORE_NUM_NEIGHBORS = 9
PATCHCORE_POOL_KERNEL_SIZE = 3
MODEL_RUN_NAME = f"patchcore_{PATCHCORE_BACKBONE}"

# PatchCore fits a memory bank without gradient-based optimization.
OPTIMIZER_NAME = "none"
SCHEDULER_NAME = "none"

# Temporary model-specific direct-resize baseline.
IMAGE_SIZE = (512, 512)
NORMALIZE_MEAN = (0.485, 0.456, 0.406)
NORMALIZE_STD = (0.229, 0.224, 0.225)
TRAIN_AUGMENTATIONS_ENABLED = False
TRAIN_BRIGHTNESS = 0.08
TRAIN_CONTRAST = 0.08
TRAIN_GAMMA = 0.08
TRAIN_SATURATION = 0.06
TRAIN_SENSOR_NOISE = 0.004
TRAIN_GAUSSIAN_NOISE = 0.004
TRAIN_GAUSSIAN_BLUR = (0.1, 0.5)

# Essential threshold-free evaluation metrics.
ESSENTIAL_METRIC_NAMES = (
    "image_auroc",
    "image_average_precision",
    "pixel_auroc",
    "pixel_average_precision",
)
METRIC_HISTOGRAM_BINS = 2048
RESTRICT_PIXELS_TO_TARGET_MASK = False

# Optional Google Drive persistence through Kaggle Secrets.
DRIVE_UPLOAD_ENABLED = False
DRIVE_PARENT_FOLDER_ID = None
DRIVE_UPLOAD_CHECKPOINTS = True
RUN_ID = (
    f"{MODEL_RUN_NAME}__seed{SEED}__"
    f"{datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')}"
)
OUTPUT_DIR = Path("/kaggle/working") / RUN_ID
OUTPUT_DIR.mkdir(parents=True, exist_ok=False)

# Visual preprocessing audit.
AUDIT_SPLIT = "train"
AUDIT_INDICES = (0, 1, 2)
AUDIT_VARIANTS = 2

# Expected contract for dataset version v1_10000.
EXPECTED_SPLIT_CONDITION_COUNTS = Counter({
    ("train", "clean"): 7000,
    ("validation", "clean"): 750,
    ("validation", "hole"): 250,
    ("test", "clean"): 1000,
    ("test", "hole"): 1000,
})

In [ ]:
@dataclass(frozen=True)
class PreprocessingConfig:
    # resize uses (height, width). None disables each optional effect.
    resize: tuple[int, int] | None = None
    normalize_mean: tuple[float, float, float] | None = None
    normalize_std: tuple[float, float, float] | None = None
    augmentations_enabled: bool = True
    brightness: float | None = None
    contrast: float | None = None
    gamma: float | None = None
    saturation: float | None = None
    sensor_noise: float | None = None
    gaussian_noise: float | None = None
    gaussian_blur: tuple[float, float] | None = None

    def __post_init__(self) -> None:
        if self.resize is not None:
            if len(self.resize) != 2 or any(value < 1 for value in self.resize):
                raise ValueError("resize must be a positive (height, width) pair")
        if (self.normalize_mean is None) != (self.normalize_std is None):
            raise ValueError("normalize_mean and normalize_std must be set together")
        if self.normalize_mean is not None:
            if len(self.normalize_mean) != 3 or len(self.normalize_std or ()) != 3:
                raise ValueError("normalization mean and std must contain three RGB values")
            if any(value <= 0 for value in self.normalize_std or ()):
                raise ValueError("normalization std values must be positive")
        for name in ("brightness", "contrast", "gamma", "saturation", "sensor_noise", "gaussian_noise"):
            value = getattr(self, name)
            if value is not None and value < 0:
                raise ValueError(f"{name} must be non-negative or None")
        for name in ("brightness", "contrast", "gamma", "saturation"):
            value = getattr(self, name)
            if value is not None and value >= 1:
                raise ValueError(f"{name} must be smaller than 1")
        if self.gaussian_blur is not None:
            if len(self.gaussian_blur) != 2:
                raise ValueError("gaussian_blur must be a (min_sigma, max_sigma) pair")
            minimum, maximum = self.gaussian_blur
            if minimum <= 0 or maximum < minimum:
                raise ValueError("gaussian_blur requires 0 < min_sigma <= max_sigma")

class WheelPreprocessor:
    def __init__(self, config: PreprocessingConfig | None = None) -> None:
        self.config = config or PreprocessingConfig()

    @staticmethod
    def _symmetric_factor(maximum_delta: float) -> float:
        return 1.0 + (2.0 * torch.rand(1).item() - 1.0) * maximum_delta

    def __call__(self, image, target_mask, anomaly_mask):
        config = self.config
        if config.resize is not None:
            image = TF.resize(image, config.resize, InterpolationMode.BILINEAR, antialias=True)
            target_mask = TF.resize(target_mask, config.resize, InterpolationMode.NEAREST)
            anomaly_mask = TF.resize(anomaly_mask, config.resize, InterpolationMode.NEAREST)
        # Stable model-facing contract for every split: float32 in [0, 1]
        # before optional normalization. Masks remain uint8.
        image = TF.convert_image_dtype(image, torch.float32)
        if config.augmentations_enabled:
            if config.brightness is not None:
                image = TF.adjust_brightness(image, self._symmetric_factor(config.brightness))
            if config.contrast is not None:
                image = TF.adjust_contrast(image, self._symmetric_factor(config.contrast))
            if config.gamma is not None:
                image = TF.adjust_gamma(image, self._symmetric_factor(config.gamma))
            if config.saturation is not None:
                image = TF.adjust_saturation(image, self._symmetric_factor(config.saturation))
            if config.gaussian_blur is not None:
                minimum, maximum = config.gaussian_blur
                sigma = minimum + torch.rand(1).item() * (maximum - minimum)
                kernel_size = 2 * max(1, round(3.0 * maximum)) + 1
                image = TF.gaussian_blur(image, kernel_size, sigma)
            if config.sensor_noise is not None and config.sensor_noise > 0:
                image = image + torch.randn_like(image) * image.clamp(0, 1).sqrt() * config.sensor_noise
            if config.gaussian_noise is not None and config.gaussian_noise > 0:
                image = image + torch.randn_like(image) * config.gaussian_noise
        image = image.clamp(0, 1)
        if config.normalize_mean is not None:
            image = TF.normalize(image, config.normalize_mean, config.normalize_std)
        return image, target_mask, anomaly_mask

    def image_for_display(self, image: torch.Tensor) -> torch.Tensor:
        if image.dtype == torch.uint8:
            return TF.convert_image_dtype(image, torch.float32)
        result = image.detach().clone()
        if self.config.normalize_mean is not None:
            mean = result.new_tensor(self.config.normalize_mean).view(3, 1, 1)
            std = result.new_tensor(self.config.normalize_std).view(3, 1, 1)
            result = result * std + mean
        return result.clamp(0, 1)

In [ ]:
# Kaggle exposes attached datasets as extracted directories.
def find_dataset_root(input_root: Path = KAGGLE_INPUT) -> Path:
    candidates = [
        manifest.parent
        for manifest in input_root.rglob("samples.csv")
        if (manifest.parent / "images").is_dir()
        and (manifest.parent / "masks").is_dir()
    ]
    if not candidates:
        raise FileNotFoundError(
            f"No valid dataset found under {input_root}: "
            "samples.csv, images/, and masks/ are required."
        )
    if len(candidates) > 1:
        raise RuntimeError(f"Multiple dataset candidates found: {candidates}")
    return candidates[0]


DATASET_ROOT = find_dataset_root()
print(f"[1/3] Dataset found: {DATASET_ROOT}")

In [ ]:
# Self-contained copy of the loader used by the terminal script.
REQUIRED_COLUMNS = {
    "image_id",
    "split",
    "condition",
    "image_path",
    "target_mask_path",
    "anomaly_mask_path",
    "pair_id",
}
VALID_SPLITS = ("train", "validation", "test")
VALID_CONDITIONS = {"clean", "hole"}
ARTIFACT_PATH_FIELDS = ("image_path", "target_mask_path", "anomaly_mask_path")


class CuriosityWheelDataset(Dataset[dict[str, Any]]):
    """Load one split of the extracted Curiosity wheel dataset."""

    def __init__(
        self,
        root: str | Path,
        split: str,
        preprocessing: WheelPreprocessor | None = None,
    ) -> None:
        self.root = Path(root).expanduser().resolve()
        self.split = split
        self.preprocessing = preprocessing or WheelPreprocessor()

        if split not in VALID_SPLITS:
            raise ValueError(f"Unsupported split {split!r}; expected one of {VALID_SPLITS}")
        if not self.root.is_dir():
            raise NotADirectoryError(f"Dataset root is not a directory: {self.root}")

        manifest_path = self.root / "samples.csv"
        if not manifest_path.is_file():
            raise FileNotFoundError(f"Dataset manifest is missing: {manifest_path}")

        with manifest_path.open(encoding="utf-8", newline="") as stream:
            reader = csv.DictReader(stream)
            missing = REQUIRED_COLUMNS - set(reader.fieldnames or [])
            if missing:
                raise ValueError(f"samples.csv is missing required columns: {sorted(missing)}")
            rows = list(reader)

        self._validate_manifest_rows(rows)
        self.rows = [row for row in rows if row["split"] == split]

        if not self.rows:
            raise ValueError(f"No samples found for split {split!r}")

        for row in self.rows:
            for field in ARTIFACT_PATH_FIELDS:
                relative_path = row[field]
                if relative_path and not (self.root / relative_path).is_file():
                    raise FileNotFoundError(
                        f"Dataset artifact is missing: {self.root / relative_path}"
                    )

    @classmethod
    def _validate_manifest_rows(cls, rows: list[dict[str, str]]) -> None:
        if not rows:
            raise ValueError("samples.csv contains no samples")
        image_ids: set[str] = set()
        pairs: dict[str, list[dict[str, str]]] = {}
        for row in rows:
            image_id = row["image_id"]
            if not image_id:
                raise ValueError("samples.csv contains an empty image_id")
            if image_id in image_ids:
                raise ValueError(f"Duplicate image_id in samples.csv: {image_id}")
            image_ids.add(image_id)
            row_split = row["split"]
            if row_split not in VALID_SPLITS:
                raise ValueError(
                    f"Unsupported split {row_split!r} in sample {image_id}; "
                    f"expected one of {VALID_SPLITS}"
                )
            condition = row["condition"]
            if condition not in VALID_CONDITIONS:
                raise ValueError(f"Unsupported condition {condition!r} in sample {image_id}")
            if not row["image_path"] or not row["target_mask_path"]:
                raise ValueError(f"Sample has incomplete artifact paths: {image_id}")
            if condition == "hole" and not row["anomaly_mask_path"]:
                raise ValueError(f"Hole sample has no anomaly mask: {image_id}")
            if condition == "clean" and row["anomaly_mask_path"]:
                raise ValueError(f"Clean sample unexpectedly has an anomaly mask: {image_id}")
            for field in ARTIFACT_PATH_FIELDS:
                if row[field]:
                    cls._validate_relative_path(row[field])
            pair_id = row["pair_id"]
            if condition == "hole" and not pair_id:
                raise ValueError(f"Hole sample has no pair_id: {image_id}")
            if pair_id:
                pairs.setdefault(pair_id, []).append(row)

        for pair_id, pair_rows in pairs.items():
            conditions = {row["condition"] for row in pair_rows}
            splits = {row["split"] for row in pair_rows}
            if len(pair_rows) != 2 or conditions != VALID_CONDITIONS:
                raise ValueError(
                    f"Pair {pair_id!r} must contain exactly one clean and one hole sample"
                )
            if len(splits) != 1:
                raise ValueError(f"Pair {pair_id!r} crosses dataset splits: {sorted(splits)}")

    @staticmethod
    def _validate_relative_path(value: str) -> None:
        path = Path(value)
        if path.is_absolute() or ".." in path.parts:
            raise ValueError(f"Invalid dataset-relative path: {value!r}")

    def _load_image(self, relative_path: str, mode: str) -> Image.Image:
        path = self.root / relative_path
        if not path.is_file():
            raise FileNotFoundError(f"Dataset artifact is missing: {path}")
        with Image.open(path) as image:
            return image.convert(mode)

    def __len__(self) -> int:
        return len(self.rows)

    def __getitem__(self, index: int) -> dict[str, Any]:
        row = self.rows[index]
        image = pil_to_tensor(self._load_image(row["image_path"], "RGB"))
        target_mask = pil_to_tensor(self._load_image(row["target_mask_path"], "L"))

        if image.shape[1:] != target_mask.shape[1:]:
            raise ValueError(f"Image/target mask size mismatch for {row['image_id']}")

        anomaly_path = row["anomaly_mask_path"]
        if anomaly_path:
            anomaly_mask = pil_to_tensor(self._load_image(anomaly_path, "L"))
            if anomaly_mask.shape != target_mask.shape:
                raise ValueError(f"Target/anomaly mask size mismatch for {row['image_id']}")
        else:
            # Clean samples use an empty mask so every batch has one stable schema.
            anomaly_mask = torch.zeros_like(target_mask)

        image, target_mask, anomaly_mask = self.preprocessing(
            image, target_mask, anomaly_mask
        )

        return {
            "image": image,
            "target_mask": target_mask,
            "anomaly_mask": anomaly_mask,
            "label": torch.tensor(row["condition"] == "hole", dtype=torch.long),
            "has_anomaly_mask": bool(anomaly_path),
            "metadata": {
                key: value
                for key, value in row.items()
                if key not in {"image_path", "target_mask_path", "anomaly_mask_path"}
            },
        }

In [ ]:
def build_dataloader(
    root: str | Path,
    split: str,
    *,
    batch_size: int = 4,
    num_workers: int = 0,
    pin_memory: bool = False,
    seed: int = 42,
    preprocessing: WheelPreprocessor | None = None,
) -> DataLoader:
    if batch_size < 1:
        raise ValueError("batch_size must be at least 1")
    if num_workers < 0:
        raise ValueError("num_workers cannot be negative")

    dataset = CuriosityWheelDataset(root, split, preprocessing=preprocessing)
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=split == "train",
        num_workers=num_workers,
        pin_memory=pin_memory,
        persistent_workers=num_workers > 0,
        generator=torch.Generator().manual_seed(seed),
    )


def build_dataloaders(
    root: str | Path,
    *,
    batch_size: int = 4,
    num_workers: int = 0,
    pin_memory: bool = False,
    seed: int = 42,
    train_preprocessing: WheelPreprocessor | None = None,
    evaluation_preprocessing: WheelPreprocessor | None = None,
) -> tuple[DataLoader, DataLoader, DataLoader]:
    def make_loader(split: str) -> DataLoader:
        return build_dataloader(
            root,
            split,
            batch_size=batch_size,
            num_workers=num_workers,
            pin_memory=pin_memory,
            seed=seed,
            preprocessing=train_preprocessing if split == "train" else evaluation_preprocessing,
        )

    return (
        make_loader("train"),
        make_loader("validation"),
        make_loader("test"),
    )

In [ ]:
@dataclass(frozen=True)
class AnomalyPrediction:
    anomaly_score: torch.Tensor
    anomaly_map: torch.Tensor

    def __post_init__(self):
        if self.anomaly_score.ndim != 1:
            raise ValueError("anomaly_score must have shape [B]")
        if self.anomaly_map.ndim != 4 or self.anomaly_map.shape[1] != 1:
            raise ValueError("anomaly_map must have shape [B, 1, H, W]")
        if self.anomaly_score.shape[0] != self.anomaly_map.shape[0]:
            raise ValueError("anomaly_score and anomaly_map batch sizes must match")
        for name, value in (("anomaly_score", self.anomaly_score), ("anomaly_map", self.anomaly_map)):
            if not value.is_floating_point() or not torch.isfinite(value).all():
                raise ValueError(f"{name} must contain finite floating-point values")
            if value.numel() and (value.min() < 0 or value.max() > 1):
                raise ValueError(f"{name} must be normalized to [0, 1]")


class AnomalyDetector(nn.Module):
    @property
    def is_fitted(self):
        raise NotImplementedError

    def fit(self, train_loader: Iterable[Mapping[str, Any]], *, device):
        raise NotImplementedError

    @torch.no_grad()
    def predict(self, images):
        raise NotImplementedError

    def checkpoint_config(self):
        return {}

    def save(self, path, *, metadata=None):
        path = Path(path)
        path.parent.mkdir(parents=True, exist_ok=True)
        torch.save({
            "schema_version": 2,
            "model_class": f"{type(self).__module__}.{type(self).__qualname__}",
            "model_config": self.checkpoint_config(),
            "model_state_dict": self.state_dict(),
            "metadata": dict(metadata or {}),
        }, path)
        return path

    def _prepare_state_dict_for_load(self, state_dict):
        pass

    def load(self, path, *, map_location=None):
        payload = torch.load(Path(path), map_location=map_location)
        if not isinstance(payload, dict) or "model_state_dict" not in payload:
            raise ValueError("Checkpoint does not contain model_state_dict")
        if payload.get("schema_version") != 2:
            raise ValueError(f"Unsupported checkpoint schema {payload.get('schema_version')!r}")
        expected_class = f"{type(self).__module__}.{type(self).__qualname__}"
        if payload.get("model_class") != expected_class:
            raise ValueError(
                f"Checkpoint model class {payload.get('model_class')!r} does not "
                f"match {expected_class!r}"
            )
        saved_config = payload.get("model_config")
        expected_config = self.checkpoint_config()
        if saved_config != expected_config:
            raise ValueError(
                "Checkpoint model configuration does not match the current model: "
                f"saved={saved_config!r}, current={expected_config!r}"
            )
        state_dict = payload["model_state_dict"]
        self._prepare_state_dict_for_load(state_dict)
        self.load_state_dict(state_dict)
        return dict(payload.get("metadata", {}))

In [ ]:
class PatchCore(AnomalyDetector):
    # This milestone extracts patch embeddings only. Memory-bank fitting
    # and nearest-neighbour anomaly scoring will be added next.
    def __init__(
        self,
        *,
        backbone="resnet18",
        pretrained=True,
        layers=("layer2", "layer3"),
        coreset_sampling_ratio=0.1,
        num_neighbors=9,
        pool_kernel_size=3,
    ):
        super().__init__()
        if not layers:
            raise ValueError("layers must contain at least one feature node")
        if not 0 < coreset_sampling_ratio <= 1:
            raise ValueError("coreset_sampling_ratio must be in (0, 1]")
        if num_neighbors < 1:
            raise ValueError("num_neighbors must be at least 1")
        if pool_kernel_size < 1 or pool_kernel_size % 2 == 0:
            raise ValueError("pool_kernel_size must be a positive odd integer")

        weights = get_model_weights(backbone).DEFAULT if pretrained else None
        backbone_model = get_model(backbone, weights=weights)
        self.feature_extractor = create_feature_extractor(
            backbone_model,
            return_nodes={layer: layer for layer in layers},
        )
        self.feature_extractor.requires_grad_(False)
        self.backbone = backbone
        self.pretrained = pretrained
        self.layers = tuple(layers)
        self.coreset_sampling_ratio = float(coreset_sampling_ratio)
        self.num_neighbors = int(num_neighbors)
        self.pool_kernel_size = int(pool_kernel_size)
        self.register_buffer("memory_bank", torch.empty(0), persistent=True)
        self.train(False)

    @property
    def is_fitted(self):
        return self.memory_bank.numel() > 0

    def checkpoint_config(self):
        return {
            "backbone": self.backbone,
            "layers": list(self.layers),
            "coreset_sampling_ratio": self.coreset_sampling_ratio,
            "num_neighbors": self.num_neighbors,
            "pool_kernel_size": self.pool_kernel_size,
        }

    def train(self, mode=True):
        # The frozen backbone must never update BatchNorm statistics.
        super().train(False)
        return self

    def forward(self, images):
        features = self.feature_extractor(images)
        target_height = max(feature.shape[-2] for feature in features.values())
        target_width = max(feature.shape[-1] for feature in features.values())
        aggregated = []
        for layer in self.layers:
            feature = F.avg_pool2d(
                features[layer],
                kernel_size=self.pool_kernel_size,
                stride=1,
                padding=self.pool_kernel_size // 2,
            )
            if feature.shape[-2:] != (target_height, target_width):
                feature = F.interpolate(
                    feature,
                    size=(target_height, target_width),
                    mode="bilinear",
                    align_corners=False,
                )
            aggregated.append(feature)
        return torch.cat(aggregated, dim=1)

    def _prepare_state_dict_for_load(self, state_dict):
        memory_bank = state_dict.get("memory_bank")
        if memory_bank is not None and memory_bank.shape != self.memory_bank.shape:
            self.memory_bank = torch.empty_like(memory_bank)


def build_model():
    if MODEL_NAME == "patchcore":
        return PatchCore(
            backbone=PATCHCORE_BACKBONE,
            pretrained=PATCHCORE_PRETRAINED,
            layers=PATCHCORE_LAYERS,
            coreset_sampling_ratio=PATCHCORE_CORESET_SAMPLING_RATIO,
            num_neighbors=PATCHCORE_NUM_NEIGHBORS,
            pool_kernel_size=PATCHCORE_POOL_KERNEL_SIZE,
        )
    raise ValueError(f"Unknown model name: {MODEL_NAME}")


def build_optimizer(model):
    if OPTIMIZER_NAME == "none":
        if any(parameter.requires_grad for parameter in model.parameters()):
            raise ValueError("optimizer=none requires a fully frozen model")
        return None
    raise ValueError("PatchCore requires OPTIMIZER_NAME='none'")


def build_scheduler(optimizer):
    if SCHEDULER_NAME == "none":
        return None
    if optimizer is None:
        raise ValueError("A scheduler requires an optimizer")
    raise ValueError("PatchCore requires SCHEDULER_NAME='none'")


model = build_model()
optimizer = build_optimizer(model)
scheduler = build_scheduler(optimizer)
total_parameters = sum(parameter.numel() for parameter in model.parameters())
trainable_parameters = sum(
    parameter.numel() for parameter in model.parameters() if parameter.requires_grad
)
print(
    f"Model: {MODEL_NAME} | backbone={PATCHCORE_BACKBONE} | "
    f"parameters={total_parameters:,} | trainable={trainable_parameters:,} | "
    f"optimizer={OPTIMIZER_NAME} | scheduler={SCHEDULER_NAME}"
)

In [ ]:
class ExactBinaryMetrics:
    # Image-level evaluation retains only one score and label per image.
    def __init__(self):
        self._scores = []
        self._targets = []

    @torch.no_grad()
    def update(self, scores, targets):
        scores = scores.detach().reshape(-1).cpu()
        targets = targets.detach().reshape(-1).cpu().bool()
        if scores.numel() != targets.numel():
            raise ValueError("scores and targets must contain the same number of values")
        if not scores.numel():
            return
        if not scores.is_floating_point() or not torch.isfinite(scores).all():
            raise ValueError("scores must contain finite floating-point values")
        if scores.min() < 0 or scores.max() > 1:
            raise ValueError("scores must be normalized to [0, 1]")
        self._scores.append(scores.clone())
        self._targets.append(targets.clone())

    def compute(self):
        if not self._scores:
            raise ValueError("AUROC and average precision require both target classes")
        scores = torch.cat(self._scores)
        targets = torch.cat(self._targets)
        positives = int(targets.sum())
        negatives = targets.numel() - positives
        if positives == 0 or negatives == 0:
            raise ValueError("AUROC and average precision require both target classes")
        order = torch.argsort(scores, descending=True)
        sorted_scores = scores[order]
        sorted_targets = targets[order]
        _, group_counts = torch.unique_consecutive(sorted_scores, return_counts=True)
        group_ends = group_counts.cumsum(0) - 1
        true_positives = sorted_targets.cumsum(0)[group_ends].double()
        false_positives = (~sorted_targets).cumsum(0)[group_ends].double()
        recall = true_positives / positives
        false_positive_rate = false_positives / negatives
        precision = true_positives / (true_positives + false_positives)
        zero = torch.zeros(1, dtype=torch.float64)
        auroc = torch.trapezoid(
            torch.cat((zero, recall)), torch.cat((zero, false_positive_rate))
        )
        recall_increment = recall - torch.cat((zero, recall[:-1]))
        average_precision = (precision * recall_increment).sum()
        return {"auroc": float(auroc), "average_precision": float(average_precision)}


class BinaryHistogramMetrics:
    # Fixed-size histograms avoid retaining every pixel score in memory.
    def __init__(self, num_bins=2048):
        if num_bins < 2:
            raise ValueError("num_bins must be at least 2")
        self.num_bins = int(num_bins)
        self.positive_histogram = torch.zeros(num_bins, dtype=torch.int64)
        self.negative_histogram = torch.zeros(num_bins, dtype=torch.int64)

    @torch.no_grad()
    def update(self, scores, targets, valid_mask=None):
        scores = scores.detach().reshape(-1).cpu()
        targets = targets.detach().reshape(-1).cpu().bool()
        if scores.numel() != targets.numel():
            raise ValueError("scores and targets must contain the same number of values")
        if valid_mask is not None:
            valid_mask = valid_mask.detach().reshape(-1).cpu().bool()
            if valid_mask.numel() != scores.numel():
                raise ValueError("valid_mask must match scores")
            scores = scores[valid_mask]
            targets = targets[valid_mask]
        if not scores.numel():
            return
        if not scores.is_floating_point() or not torch.isfinite(scores).all():
            raise ValueError("scores must contain finite floating-point values")
        if scores.min() < 0 or scores.max() > 1:
            raise ValueError("scores must be normalized to [0, 1]")
        bins = (scores * self.num_bins).long().clamp(max=self.num_bins - 1)
        self.positive_histogram += torch.bincount(bins[targets], minlength=self.num_bins)
        self.negative_histogram += torch.bincount(bins[~targets], minlength=self.num_bins)

    def compute(self):
        positives = int(self.positive_histogram.sum())
        negatives = int(self.negative_histogram.sum())
        if positives == 0 or negatives == 0:
            raise ValueError("AUROC and average precision require both target classes")
        true_positives = self.positive_histogram.flip(0).cumsum(0).double()
        false_positives = self.negative_histogram.flip(0).cumsum(0).double()
        recall = true_positives / positives
        false_positive_rate = false_positives / negatives
        precision = true_positives / (true_positives + false_positives).clamp_min(1)
        zero = torch.zeros(1, dtype=torch.float64)
        auroc = torch.trapezoid(
            torch.cat((zero, recall)), torch.cat((zero, false_positive_rate))
        )
        recall_increment = recall - torch.cat((zero, recall[:-1]))
        average_precision = (precision * recall_increment).sum()
        return {"auroc": float(auroc), "average_precision": float(average_precision)}


class AnomalyMetrics:
    def __init__(self, histogram_bins=2048):
        self.image = ExactBinaryMetrics()
        self.pixel = BinaryHistogramMetrics(histogram_bins)

    @torch.no_grad()
    def update(self, prediction, labels, anomaly_masks, *, valid_pixel_mask=None):
        labels = labels.reshape(-1)
        if labels.shape != prediction.anomaly_score.shape:
            raise ValueError("labels must match anomaly_score shape")
        anomaly_masks = anomaly_masks > 0
        if anomaly_masks.ndim == 3:
            anomaly_masks = anomaly_masks.unsqueeze(1)
        if anomaly_masks.shape != prediction.anomaly_map.shape:
            raise ValueError("anomaly_masks must match anomaly_map shape")
        if valid_pixel_mask is not None:
            valid_pixel_mask = valid_pixel_mask > 0
            if valid_pixel_mask.ndim == 3:
                valid_pixel_mask = valid_pixel_mask.unsqueeze(1)
            if valid_pixel_mask.shape != prediction.anomaly_map.shape:
                raise ValueError("valid_pixel_mask must match anomaly_map shape")
        self.image.update(prediction.anomaly_score, labels)
        self.pixel.update(prediction.anomaly_map, anomaly_masks, valid_pixel_mask)

    def compute(self):
        image = self.image.compute()
        pixel = self.pixel.compute()
        return {
            "image_auroc": image["auroc"],
            "image_average_precision": image["average_precision"],
            "pixel_auroc": pixel["auroc"],
            "pixel_average_precision": pixel["average_precision"],
        }


metrics_accumulator = AnomalyMetrics(METRIC_HISTOGRAM_BINS)

def update_metrics_from_batch(metrics, prediction, batch):
    valid_pixel_mask = (
        batch["target_mask"] if RESTRICT_PIXELS_TO_TARGET_MASK else None
    )
    metrics.update(
        prediction, batch["label"], batch["anomaly_mask"],
        valid_pixel_mask=valid_pixel_mask,
    )

In [ ]:
# Build preprocessing from the variables declared in the configuration cell.
EVALUATION_PREPROCESSING_CONFIG = PreprocessingConfig(
    resize=IMAGE_SIZE,
    normalize_mean=NORMALIZE_MEAN,
    normalize_std=NORMALIZE_STD,
)
EVALUATION_PREPROCESSING = WheelPreprocessor(EVALUATION_PREPROCESSING_CONFIG)

TRAIN_PREPROCESSING_CONFIG = PreprocessingConfig(
    resize=IMAGE_SIZE,
    normalize_mean=NORMALIZE_MEAN,
    normalize_std=NORMALIZE_STD,
    augmentations_enabled=TRAIN_AUGMENTATIONS_ENABLED,
    brightness=TRAIN_BRIGHTNESS,
    contrast=TRAIN_CONTRAST,
    gamma=TRAIN_GAMMA,
    saturation=TRAIN_SATURATION,
    sensor_noise=TRAIN_SENSOR_NOISE,
    gaussian_noise=TRAIN_GAUSSIAN_NOISE,
    gaussian_blur=TRAIN_GAUSSIAN_BLUR,
)
TRAIN_PREPROCESSING = WheelPreprocessor(TRAIN_PREPROCESSING_CONFIG)

# Build the three DataLoaders. Stochastic augmentation is train-only.
train_loader, validation_loader, test_loader = build_dataloaders(
    DATASET_ROOT,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    seed=SEED,
    train_preprocessing=TRAIN_PREPROCESSING,
    evaluation_preprocessing=EVALUATION_PREPROCESSING,
)

loaders = (
    ("train", train_loader),
    ("validation", validation_loader),
    ("test", test_loader),
)
for split, loader in loaders:
    print(f"{split:10s}: {len(loader.dataset)} samples, {len(loader)} batches")

observed_counts = Counter(
    (row["split"], row["condition"])
    for _, loader in loaders
    for row in loader.dataset.rows
)
if observed_counts != EXPECTED_SPLIT_CONDITION_COUNTS:
    raise ValueError(
        f"Unexpected split/condition counts: {dict(observed_counts)}; "
        f"expected {dict(EXPECTED_SPLIT_CONDITION_COUNTS)}"
    )
print("[2/3] DataLoaders ready")

In [ ]:
def audit_preprocessing(
    root: str | Path,
    config: PreprocessingConfig,
    *,
    split: str = "train",
    indices: Sequence[int] | None = None,
    variants: int = 2,
    seed: int = 42,
) -> plt.Figure:
    if variants < 1:
        raise ValueError("variants must be at least 1")
    dataset = CuriosityWheelDataset(root, split)
    selected = list(indices) if indices is not None else list(range(min(4, len(dataset))))
    if not selected:
        raise ValueError("indices must select at least one sample")
    if any(index < 0 or index >= len(dataset) for index in selected):
        raise IndexError("audit sample index is outside the dataset")

    preprocessor = WheelPreprocessor(config)
    raw_preprocessor = WheelPreprocessor()
    figure, axes = plt.subplots(
        len(selected), variants + 1,
        figsize=(4.5 * (variants + 1), 3.5 * len(selected)),
        squeeze=False,
    )
    with torch.random.fork_rng():
        torch.manual_seed(seed)
        for row_index, sample_index in enumerate(selected):
            sample = dataset[sample_index]
            raw_image = sample["image"]
            target_mask = sample["target_mask"]
            anomaly_mask = sample["anomaly_mask"]
            image_id = sample["metadata"]["image_id"]
            axes[row_index, 0].imshow(
                raw_preprocessor.image_for_display(raw_image).permute(1, 2, 0)
            )
            axes[row_index, 0].set_title(f"Originale\n{image_id}")
            for variant in range(variants):
                processed, _, _ = preprocessor(
                    raw_image.clone(), target_mask.clone(), anomaly_mask.clone()
                )
                axes[row_index, variant + 1].imshow(
                    preprocessor.image_for_display(processed).permute(1, 2, 0)
                )
                axes[row_index, variant + 1].set_title(f"Augmentata {variant + 1}")
            for axis in axes[row_index]:
                axis.axis("off")
    figure.suptitle(f"Audit preprocessing — split {split}")
    figure.tight_layout()
    return figure


audit_figure = audit_preprocessing(
    DATASET_ROOT,
    TRAIN_PREPROCESSING_CONFIG,
    split=AUDIT_SPLIT,
    indices=AUDIT_INDICES,
    variants=AUDIT_VARIANTS,
    seed=SEED,
)
plt.show()

In [ ]:
# Smoke test: verify the common numerical contract on one batch per split.
for split, loader in loaders:
    batch = next(iter(loader))
    if batch["image"].dtype != torch.float32 or not torch.isfinite(batch["image"]).all():
        raise TypeError(f"{split} images must be finite float32 tensors")
    if batch["target_mask"].dtype != torch.uint8 or batch["anomaly_mask"].dtype != torch.uint8:
        raise TypeError(f"{split} masks must remain uint8 tensors")
    print(
        f"{split:10s} image={tuple(batch['image'].shape)} {batch['image'].dtype} "
        f"range=[{batch['image'].min().item():.3f}, {batch['image'].max().item():.3f}] "
        f"target_mask={tuple(batch['target_mask'].shape)} "
        f"anomaly_mask={tuple(batch['anomaly_mask'].shape)} "
        f"label={batch['label'].tolist()}"
    )
print("[3/3] Loading verified")

In [ ]:
# Save locally first, then optionally copy the same run artifacts to Drive.
DRIVE_SCOPE = "https://www.googleapis.com/auth/drive.file"
DRIVE_FOLDER_MIME = "application/vnd.google-apps.folder"

def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        while chunk := handle.read(chunk_size):
            digest.update(chunk)
    return digest.hexdigest()

def save_json_atomic(data, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(
        json.dumps(data, indent=2, sort_keys=True, default=str) + "\n",
        encoding="utf-8",
    )
    os.replace(temporary, path)
    return path

def get_kaggle_secret(name, required=False):
    try:
        from kaggle_secrets import UserSecretsClient
        value = UserSecretsClient().get_secret(name)
    except Exception:
        value = None
    if required and not value:
        raise RuntimeError(f"Missing required Kaggle secret: {name}")
    return value

def build_drive_service():
    from google.oauth2.credentials import Credentials
    from googleapiclient.discovery import build
    credentials = Credentials(
        token=None,
        refresh_token=get_kaggle_secret("GDRIVE_REFRESH_TOKEN", required=True),
        token_uri="https://oauth2.googleapis.com/token",
        client_id=get_kaggle_secret("GDRIVE_CLIENT_ID", required=True),
        client_secret=get_kaggle_secret("GDRIVE_CLIENT_SECRET", required=True),
        scopes=[DRIVE_SCOPE],
    )
    return build("drive", "v3", credentials=credentials, cache_discovery=False)

def drive_children(service, parent_id, name=None):
    escaped_parent = parent_id.replace("'", "\\'")
    query = [f"'{escaped_parent}' in parents", "trashed = false"]
    if name is not None:
        escaped_name = name.replace("\\", "\\\\").replace("'", "\\'")
        query.append(f"name = '{escaped_name}'")
    return service.files().list(
        q=" and ".join(query),
        spaces="drive",
        fields="files(id,name,mimeType,size)",
        pageSize=1000,
    ).execute(num_retries=3).get("files", [])

def ensure_drive_folder(service, parent_id, name, fail_if_exists=False):
    matches = [
        item for item in drive_children(service, parent_id, name)
        if item["mimeType"] == DRIVE_FOLDER_MIME
    ]
    if matches:
        if fail_if_exists:
            raise FileExistsError(f"Drive folder already exists: {name}")
        if len(matches) > 1:
            raise RuntimeError(f"Multiple Drive folders named {name}")
        return matches[0]["id"]
    return service.files().create(
        body={"name": name, "mimeType": DRIVE_FOLDER_MIME, "parents": [parent_id]},
        fields="id",
    ).execute(num_retries=3)["id"]

def drive_upload_file(service, path, parent_id):
    from googleapiclient.http import MediaFileUpload
    path = Path(path)
    media = MediaFileUpload(
        str(path), mimetype="application/octet-stream",
        chunksize=8 * 1024 * 1024, resumable=True,
    )
    request = service.files().create(
        body={"name": path.name, "parents": [parent_id]},
        media_body=media, fields="id,name,size",
    )
    response = None
    while response is None:
        _, response = request.next_chunk(num_retries=5)
    return response

def upload_run_artifacts(paths):
    service = build_drive_service()
    parent = DRIVE_PARENT_FOLDER_ID or get_kaggle_secret("GDRIVE_FOLDER_ID", required=True)
    for folder_name in ("anomaly_detection", MODEL_RUN_NAME):
        parent = ensure_drive_folder(service, parent, folder_name)
    run_folder = ensure_drive_folder(service, parent, RUN_ID, fail_if_exists=True)
    uploaded, failures = [], []
    for path in paths:
        path = Path(path)
        if path.suffix == ".ckpt" and not DRIVE_UPLOAD_CHECKPOINTS:
            continue
        try:
            response = drive_upload_file(service, path, run_folder)
            uploaded.append({
                "name": path.name,
                "drive_file_id": response["id"],
                "size_bytes": path.stat().st_size,
                "sha256": sha256_file(path),
            })
        except Exception as error:
            failures.append({"name": path.name, "error": type(error).__name__})
    return service, {
        "status": "complete" if not failures else "partial",
        "drive_folder_id": run_folder,
        "uploaded": uploaded,
        "failures": failures,
    }

def persist_run_artifacts(detector, results):
    # Call this only after detector.fit(...) and test evaluation are implemented.
    if not detector.is_fitted:
        raise RuntimeError("Cannot save an unfitted anomaly detector")
    if set(results) != set(ESSENTIAL_METRIC_NAMES):
        raise ValueError(f"Results must contain only: {ESSENTIAL_METRIC_NAMES}")
    run_metadata = {
        "run_id": RUN_ID,
        "model_run_name": MODEL_RUN_NAME,
        "seed": SEED,
        "model": {
            "name": MODEL_NAME,
            "pretrained_initialization": PATCHCORE_PRETRAINED,
            **detector.checkpoint_config(),
        },
        "preprocessing": {
            "input_size": list(IMAGE_SIZE),
            "normalize_mean": list(NORMALIZE_MEAN),
            "normalize_std": list(NORMALIZE_STD),
            "train_augmentations_enabled": TRAIN_AUGMENTATIONS_ENABLED,
            "train_augmentation_parameters": {
                "brightness": TRAIN_BRIGHTNESS,
                "contrast": TRAIN_CONTRAST,
                "gamma": TRAIN_GAMMA,
                "saturation": TRAIN_SATURATION,
                "sensor_noise": TRAIN_SENSOR_NOISE,
                "gaussian_noise": TRAIN_GAUSSIAN_NOISE,
                "gaussian_blur": (
                    None if TRAIN_GAUSSIAN_BLUR is None
                    else list(TRAIN_GAUSSIAN_BLUR)
                ),
            },
        },
        "dataset": {
            "manifest_sha256": sha256_file(DATASET_ROOT / "samples.csv"),
            "expected_split_condition_counts": [
                {"split": split, "condition": condition, "count": count}
                for (split, condition), count
                in sorted(EXPECTED_SPLIT_CONDITION_COUNTS.items())
            ],
        },
        "evaluation": {
            "metrics": list(ESSENTIAL_METRIC_NAMES),
            "pixel_histogram_bins": METRIC_HISTOGRAM_BINS,
            "restrict_pixels_to_target_mask": RESTRICT_PIXELS_TO_TARGET_MASK,
        },
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
    }
    checkpoint_path = detector.save(OUTPUT_DIR / "model.ckpt", metadata=run_metadata)
    metrics_path = save_json_atomic(results, OUTPUT_DIR / "metrics.json")
    manifest_path = save_json_atomic({
        **run_metadata,
        "checkpoint_sha256": sha256_file(checkpoint_path),
        "metrics_sha256": sha256_file(metrics_path),
    }, OUTPUT_DIR / "run_manifest.json")
    artifact_paths = [checkpoint_path, metrics_path, manifest_path]
    upload_status = {"status": "disabled"}
    if DRIVE_UPLOAD_ENABLED:
        service, upload_status = upload_run_artifacts(artifact_paths)
        status_path = save_json_atomic(upload_status, OUTPUT_DIR / "upload_status.json")
        drive_upload_file(service, status_path, upload_status["drive_folder_id"])
    else:
        save_json_atomic(upload_status, OUTPUT_DIR / "upload_status.json")
    return {"artifacts": artifact_paths, "upload": upload_status}

print(f"Artifact output directory: {OUTPUT_DIR}")
print("Call persist_run_artifacts(model, test_metrics) after fit and evaluation.")